# Hypo3 Final: Macro vs Internal Market Factors

Final comparison test for whether broad macro/financial variables or product-market internal variables explain weekly hardware price returns better. Predictors are pre-specified from previous Hypo3 relation and internal-reg datasets; no new variable screening is done here.

In [1]:

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan

OUT = Path("outputs/hypo3_final")
PIC = Path("output_pic/hypo3_final")
OUT.mkdir(parents=True, exist_ok=True)
PIC.mkdir(parents=True, exist_ok=True)

MAC = Path("outputs/hypo3_relation/data.csv")
INT = Path("outputs/hypo3_internal_reg/int_reg.csv")
assert MAC.exists(), "Run Hypo3_relation.ipynb first."
assert INT.exists(), "Run Hypo3_internal_reg.ipynb first."


## Hypothesis

- **Macro model**: BTC, FX, SOXX, Oil, and CSI change.
- **Internal model**: active product-count change, new-entry share, lagged product-return dispersion, and compact specification-mix changes.
- **Test**: compare macro-only, internal-only, and combined models on the same cleaned weekly sample. Use HAC standard errors, VIF, residual diagnostics, adjusted R², AIC/BIC, MAE, and block Wald tests.

In [2]:

mac = pd.read_csv(MAC)
if "Unnamed: 0" in mac.columns:
    mac = mac.rename(columns={"Unnamed: 0": "date"})
mac["date"] = pd.to_datetime(mac["date"])

it = pd.read_csv(INT, parse_dates=["date"])

mx = ["btc_ret_z", "fx_ret_z", "soxx_ret_z", "oil_ret_z", "csi_chg_z"]
spec = {
    "GPU": {
        "y": "gpu_ret",
        "ix": ["gpu_n_chg", "gpu_new_sh", "gpu_riq_l1", "gpu_high_chg", "gpu_vram_chg"],
    },
    "CPU": {
        "y": "cpu_ret",
        "ix": ["cpu_n_chg", "cpu_new_sh", "cpu_riq_l1", "cpu_amd_chg"],
    },
    "RAM": {
        "y": "ram_ret",
        "ix": ["ram_n_chg", "ram_new_sh", "ram_riq_l1", "ram_ddr5_chg", "ram_cap_chg"],
    },
}

df = it.merge(mac[["date"] + mx], on="date", how="inner").sort_values("date")

xall = mx + sorted({x for v in spec.values() for x in v["ix"]})
for c in xall:
    df[c] = (df[c] - df[c].mean()) / df[c].std(ddof=0)

need = ["date"] + mx
for v in spec.values():
    need += [v["y"]] + v["ix"]
reg = df[need].dropna().reset_index(drop=True)
reg.to_csv(OUT / "reg.csv", index=False)
print(reg.shape)
reg.head()


(103, 23)


,date,btc_ret_z,fx_ret_z,soxx_ret_z,oil_ret_z,csi_chg_z,gpu_ret,gpu_n_chg,gpu_new_sh,gpu_riq_l1,...,cpu_n_chg,cpu_new_sh,cpu_riq_l1,cpu_amd_chg,ram_ret,ram_n_chg,ram_new_sh,ram_riq_l1,ram_ddr5_chg,ram_cap_chg
0,2024-03-17,0.029550,-0.055291,-0.944904,0.746887,0.143206,0.003092,-0.451920,-0.372946,-0.390757,...,-0.288861,0.422012,-0.722163,0.211714,0.001473,0.259155,0.865407,-0.353363,0.985108,1.507432
1,2024-03-24,-0.099632,0.765633,0.604315,-0.065136,0.143206,0.002050,-0.107753,-0.199170,-0.254165,...,-0.097782,0.210424,-0.783949,-0.577548,-0.003016,0.020191,1.140770,-0.351160,0.143855,0.760658
2,2024-03-31,1.013521,0.916137,-0.011723,0.420406,0.143206,0.001093,0.684418,0.336511,-0.321481,...,0.093297,1.045638,-0.797772,-0.224545,-0.001508,-0.369758,0.989665,-0.344856,-0.365304,-0.532735
3,2024-04-07,-0.382897,-0.001291,-0.462259,0.797604,-1.054228,-0.000115,-0.056597,-0.495143,-0.387381,...,-0.481068,1.706561,-0.640940,-0.718265,-0.005128,0.833641,1.067102,-0.347500,0.222338,-1.484754
4,2024-04-14,-0.526455,0.993859,-0.399126,-0.257622,0.143206,0.000292,0.090703,-0.162278,-0.356304,...,-1.076201,-0.408856,1.203490,-0.088417,0.003570,0.165022,1.447908,-0.230892,0.104270,0.625313


In [3]:

def fit(d, y, x):
    X = sm.add_constant(d[x], has_constant="add")
    return sm.OLS(d[y], X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})


def vmax(d, x):
    X = sm.add_constant(d[x], has_constant="add")
    vals = []
    for i, c in enumerate(X.columns):
        if c != "const":
            vals.append(variance_inflation_factor(X.values, i))
    return max(vals)


def diag(m):
    lb = acorr_ljungbox(m.resid, lags=[4], return_df=True)["lb_pvalue"].iloc[0]
    bp = het_breuschpagan(m.resid, m.model.exog)[1]
    return float(lb), float(bp)


def wald(m, cols):
    names = list(m.params.index)
    R = []
    for c in cols:
        r = [0] * len(names)
        r[names.index(c)] = 1
        R.append(r)
    return float(m.wald_test(np.asarray(R), scalar=True).pvalue)


def info(hw, name, m, d, y, x):
    lb, bp = diag(m)
    pred = m.fittedvalues
    return {
        "hw": hw,
        "model": name,
        "n": int(m.nobs),
        "k": len(x),
        "r2": m.rsquared,
        "adj_r2": m.rsquared_adj,
        "aic": m.aic,
        "bic": m.bic,
        "mae": np.mean(np.abs(d[y] - pred)),
        "dir_acc": (np.sign(d[y]) == np.sign(pred)).mean(),
        "max_vif": vmax(d, x),
        "lb_p": lb,
        "bp_p": bp,
    }


In [4]:

model_rows = []
coef_rows = []
test_rows = []
fits = {}

for hw, sp in spec.items():
    y = sp["y"]
    ix = sp["ix"]
    cols = ["date", y] + mx + ix
    d = reg[cols].dropna().copy()

    xs = {
        "macro": mx,
        "internal": ix,
        "combined": mx + ix,
    }
    fm = {name: fit(d, y, x) for name, x in xs.items()}
    fits[hw] = fm

    for name, m in fm.items():
        model_rows.append(info(hw, name, m, d, y, xs[name]))
        for c in xs[name]:
            coef_rows.append({
                "hw": hw,
                "model": name,
                "var": c,
                "coef": m.params[c],
                "se": m.bse[c],
                "p": m.pvalues[c],
            })

    test_rows.append({
        "hw": hw,
        "test": "internal_adds_to_macro",
        "p": wald(fm["combined"], ix),
        "adj_gain": fm["combined"].rsquared_adj - fm["macro"].rsquared_adj,
        "aic_gain": fm["macro"].aic - fm["combined"].aic,
        "mae_gain": model_rows[-3]["mae"] - model_rows[-1]["mae"],
    })
    test_rows.append({
        "hw": hw,
        "test": "macro_adds_to_internal",
        "p": wald(fm["combined"], mx),
        "adj_gain": fm["combined"].rsquared_adj - fm["internal"].rsquared_adj,
        "aic_gain": fm["internal"].aic - fm["combined"].aic,
        "mae_gain": model_rows[-2]["mae"] - model_rows[-1]["mae"],
    })

model = pd.DataFrame(model_rows)
coef = pd.DataFrame(coef_rows)
test = pd.DataFrame(test_rows)

model.to_csv(OUT / "model.csv", index=False)
coef.to_csv(OUT / "coef.csv", index=False)
test.to_csv(OUT / "test.csv", index=False)

model.round(4)


,hw,model,n,k,r2,adj_r2,aic,bic,mae,dir_acc,max_vif,lb_p,bp_p
0,GPU,macro,103,5,0.0865,0.0394,-669.9800,-654.1716,0.0052,0.4854,1.1966,0.0000,0.0179
1,GPU,internal,103,5,0.5552,0.5323,-744.1054,-728.2970,0.0041,0.6602,3.0285,0.0000,0.0000
2,GPU,combined,103,10,0.5713,0.5247,-737.9070,-708.9250,0.0042,0.6505,3.2113,0.0000,0.0002
3,CPU,macro,103,5,0.0191,-0.0315,-766.4867,-750.6783,0.0039,0.5534,1.1966,0.2262,0.9246
4,CPU,internal,103,4,0.0320,-0.0075,-769.8551,-756.6814,0.0038,0.6408,3.5427,0.6696,0.6798
5,CPU,combined,103,9,0.0486,-0.0435,-761.6324,-735.2851,0.0038,0.6117,3.5951,0.5385,0.9618
6,RAM,macro,103,5,0.0288,-0.0212,-461.8849,-446.0765,0.0138,0.5534,1.1966,0.0000,0.5806
7,RAM,internal,103,5,0.5922,0.5712,-551.2552,-535.4469,0.0098,0.5728,2.3202,0.0350,0.0000
8,RAM,combined,103,10,0.6095,0.5671,-545.7358,-516.7537,0.0102,0.5631,2.5403,0.0750,0.0000


In [5]:

# 1. Model fit comparison
fig, ax = plt.subplots(figsize=(8, 4.5))
piv = model.pivot(index="hw", columns="model", values="adj_r2").loc[["GPU", "CPU", "RAM"], ["macro", "internal", "combined"]]
piv.plot(kind="bar", ax=ax)
ax.set_title("Model comparison: adjusted R²")
ax.set_xlabel("hardware")
ax.set_ylabel("adjusted R²")
ax.axhline(0, linewidth=1)
plt.tight_layout()
plt.savefig(PIC / "01_fit.png", dpi=160)
plt.close()

# 2. Added explanatory value
fig, ax = plt.subplots(figsize=(8, 4.5))
g = test.pivot(index="hw", columns="test", values="adj_gain").loc[["GPU", "CPU", "RAM"]]
g.plot(kind="bar", ax=ax)
ax.set_title("Incremental adjusted R²")
ax.set_xlabel("hardware")
ax.set_ylabel("gain over base model")
ax.axhline(0, linewidth=1)
plt.tight_layout()
plt.savefig(PIC / "02_gain.png", dpi=160)
plt.close()

# 3. Combined-model coefficients
cm = coef[coef["model"] == "combined"].copy()
for hw in ["GPU", "CPU", "RAM"]:
    d = cm[cm["hw"] == hw].copy()
    d = d.reindex(d["coef"].abs().sort_values().index)
    fig, ax = plt.subplots(figsize=(7, 4.8))
    ax.barh(d["var"], d["coef"])
    ax.axvline(0, linewidth=1)
    ax.set_title(f"{hw}: combined model coefficients")
    ax.set_xlabel("coef on standardized predictor")
    plt.tight_layout()
    plt.savefig(PIC / f"03_coef_{hw.lower()}.png", dpi=160)
    plt.close()

print("csv:", sorted(p.name for p in OUT.glob("*.csv")))
print("plots:", sorted(p.name for p in PIC.glob("*.png")))


csv: ['coef.csv', 'model.csv', 'reg.csv', 'test.csv']
plots: ['01_fit.png', '02_gain.png', '03_coef_cpu.png', '03_coef_gpu.png', '03_coef_ram.png']


In [6]:

# Minimal validation for generated files
for p in [OUT / "reg.csv", OUT / "model.csv", OUT / "coef.csv", OUT / "test.csv"]:
    assert p.exists(), p
for p in [PIC / "01_fit.png", PIC / "02_gain.png", PIC / "03_coef_gpu.png", PIC / "03_coef_cpu.png", PIC / "03_coef_ram.png"]:
    assert p.exists(), p

print("reg rows:", len(reg))
print("missing in reg:", int(reg.isna().sum().sum()))
print("model rows:", len(model), "coef rows:", len(coef), "test rows:", len(test))
test.round(4)


reg rows: 103
missing in reg: 0
model rows: 9 coef rows: 58 test rows: 6


,hw,test,p,adj_gain,aic_gain,mae_gain
0,GPU,internal_adds_to_macro,0.0000,0.4853,67.9270,0.0010
1,GPU,macro_adds_to_internal,0.7805,-0.0075,-6.1983,-0.0001
2,CPU,internal_adds_to_macro,0.0064,-0.0120,-4.8543,0.0002
3,CPU,macro_adds_to_internal,0.8971,-0.0360,-8.2227,0.0000
4,RAM,internal_adds_to_macro,0.0000,0.5883,83.8509,0.0036
5,RAM,macro_adds_to_internal,0.6561,-0.0041,-5.5195,-0.0004
